# 05 — MCP Transports: stdio vs Streamable HTTP

MCP is a **two-layer protocol**:
- **Data layer** — JSON-RPC 2.0 messages (what they mean)
- **Transport layer** — how those messages move (the bytes on the wire)

Same data layer works over stdio (local subprocess) or Streamable HTTP (remote service). This is the same idea as ASGI vs WSGI in Python web frameworks — protocol semantics live above transport.

**By the end of this notebook you will:**
1. Run the same `notes_server` over **both** transports
2. See the `Mcp-Session-Id` header that makes Streamable HTTP stateful
3. Build a second MCP server (`cache_server`) and connect to it remotely
4. Pick the right transport for a given deployment


## 1. The decision table

| Factor | Choose **stdio** | Choose **Streamable HTTP** |
|---|---|---|
| Server location | Same machine as host | Different machine / cloud |
| Auth | Implicit (process owner) | OAuth 2.1 / API key (mandated for public servers per Nov 2025 spec) |
| Multi-tenancy | One server per host process | One server, many concurrent clients |
| Network ops | None | Reverse proxy, TLS, rate limiting available |
| Cold start | ~50-100 ms (process spawn) | ~250-600 ms (Python SDK), then warm |
| Best fit | IDE plugins, desktop apps, dev tools | Enterprise SaaS, multi-user agents, your VoyageAI-style deployment |


## 2. Same server, two transports — change one line

Look at the bottom of `notes_server.py`:

```python
if "--http" in sys.argv:
    mcp.settings.host = "0.0.0.0"; mcp.settings.port = 8765
    mcp.run(transport="streamable-http")
else:
    mcp.run()  # stdio
```

Same `@mcp.tool()` decorators. Same business logic. Just a different `transport=` argument. **Protocol decoupled from transport.**


## 3. Run notes_server as HTTP (in a background subprocess)

In [ ]:
import os
import subprocess
import sys
import time

# Kill any leftover from a previous run of this cell
subprocess.run(["pkill", "-f", "deepbrief.mcp_servers.notes_server"], capture_output=True)

notes_proc = subprocess.Popen(
    [sys.executable, "-m", "deepbrief.mcp_servers.notes_server", "--http"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    env={**os.environ, "MCP_PORT": "8766"},
)
time.sleep(2)   # give it a moment to bind the port
print(f"notes_server pid={notes_proc.pid}  url=http://localhost:8766/mcp")
print("Status:", "running" if notes_proc.poll() is None else f"DEAD (exit {notes_proc.returncode})")

## 4. Connect via Streamable HTTP

Note the difference from notebook 04: instead of `stdio_client(server_params)`, we use `streamablehttp_client(url=...)`. Everything else (initialize, list_tools, call_tool) is identical.

In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async with streamablehttp_client(url="http://localhost:8766/mcp") as (read, write, _):
    async with ClientSession(read, write) as session:
        init = await session.initialize()
        print(f"Connected via HTTP: {init.serverInfo.name}\n")

        tools = (await session.list_tools()).tools
        print(f"Tools ({len(tools)}):", [t.name for t in tools])

        # Save a note
        save = await session.call_tool(
            "save_note",
            {"title": "Saved via HTTP", "content": "Same server, different transport."},
        )
        print(f"\nsave_note → {save.content[0].text}")

        # List notes — should include the one we just saved AND notes from notebook 04
        listing = await session.call_tool("list_notes", {})
        print(f"\nlist_notes → {listing.content[0].text}")

## 5. The wire-level difference

Same JSON-RPC 2.0 messages, different envelope:

```
stdio:                              Streamable HTTP:
→  {"jsonrpc":"2.0",                 →  POST /mcp HTTP/1.1
    "method":"initialize",...}          Mcp-Session-Id: <not yet>
                                         Content-Type: application/json
                                         {"jsonrpc":"2.0","method":"initialize",...}
                                     ←  HTTP/1.1 200 OK
                                         Mcp-Session-Id: 8b7e3...   ← assigned by server
                                         {"jsonrpc":"2.0","result":{...}}
                                     →  POST /mcp
                                         Mcp-Session-Id: 8b7e3...   ← echo it back on every call
                                         {"jsonrpc":"2.0","method":"tools/list",...}
```

**Key design decision:** one endpoint (`POST /mcp`), two interaction modes.
- Simple request → JSON response (works with any HTTP infra: CDN, ALB, reverse proxy)
- Long-running tool → upgrade response to **Server-Sent Events** for streaming progress

**Sessions** are identified by the `Mcp-Session-Id` header. Stateful, but unlike WebSocket the session can survive transient disconnects — clients reconnect with the same id.

**Why not plain SSE?** The original 2024-11-05 spec used `/sse` (server→client) + `/messages` (client→server). Deprecated 2025-03-26 because: two endpoints annoyed ALB/CDN config, connection drops meant full session loss, firewalls broke long-lived SSE streams. Streamable HTTP is the current standard.

## 6. Build a second server: `cache_server`

We'll need this in notebook 06 for the agent to dedupe web fetches. It's HTTP-only by design — models the *remote shared service* pattern.

Read it briefly:

In [ ]:
import inspect
from deepbrief.mcp_servers import cache_server

src = inspect.getsource(cache_server)
print(src[:1500])
print("...\n")
print("  ↑ Pattern is exactly the same as notes_server. ")
print("    What changes: in-memory dict instead of files, HTTP-only.")

## 7. Run cache_server and connect

In [ ]:
subprocess.run(["pkill", "-f", "deepbrief.mcp_servers.cache_server"], capture_output=True)

cache_proc = subprocess.Popen(
    [sys.executable, "-m", "deepbrief.mcp_servers.cache_server"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    env={**os.environ, "PORT": "8765"},
)
time.sleep(2)
print(f"cache_server pid={cache_proc.pid}  url=http://localhost:8765/mcp")

In [ ]:
async with streamablehttp_client(url="http://localhost:8765/mcp") as (read, write, _):
    async with ClientSession(read, write) as session:
        await session.initialize()

        # Miss
        r = await session.call_tool("cache_get", {"key": "https://example.com"})
        print("miss:  ", r.content[0].text)

        # Set
        r = await session.call_tool("cache_set",
            {"key": "https://example.com", "value": "hello world"})
        print("set:   ", r.content[0].text)

        # Hit
        r = await session.call_tool("cache_get", {"key": "https://example.com"})
        print("hit:   ", r.content[0].text)

        # Stats
        r = await session.call_tool("cache_stats", {})
        print("stats: ", r.content[0].text)

## 8. Cleanup — kill the background servers

In [ ]:
for proc, name in [(notes_proc, "notes"), (cache_proc, "cache")]:
    proc.terminate()
    try:
        proc.wait(timeout=5)
        print(f"{name}_server stopped")
    except subprocess.TimeoutExpired:
        proc.kill()
        print(f"{name}_server force-killed")

## 9. The "wrap stdio as HTTP" pattern

Sometimes a server you want to use ships **only** stdio (most early MCP servers were stdio-only). But your agent runs in a Kubernetes pod and can't subprocess-spawn things. The fix is a **gateway** that runs the stdio server inside it and exposes Streamable HTTP outward.

Cloudflare's `supergateway`, `mcp-remote`, and the open-source **Bifrost** gateway all do this. We won't build one in this lab, but knowing the pattern exists is a senior signal.

## 10. Self-check

1. Same `@mcp.tool()` business logic. What changed to swap stdio → HTTP?
2. What does the `Mcp-Session-Id` header buy you over a plain WebSocket connection?
3. Why was the original Plain SSE transport deprecated in 2025?
4. Your team picked an MCP server that only ships stdio. Your agent runs in EKS. How do you bridge it?

## What's next

Notebook **06** — wire MCP tools into the agent's `ToolRegistry` via an adapter. After this notebook the agent doesn't know whether a tool is local Python, an HTTP MCP server, or a stdio MCP server — they all look the same to the loop.